# Oracle gap ratios across 50 states (Table 2 and Section 6)

Summarizes the oracle gap ratios of the single-feature models over 50 states × 2 partitions × 4 features = 400 configurations, using the CSV files written by `oracle_gap_scan_employed.ipynb`.

- Data: ACS 2018 income via folktables, full-time employees (ESR=1, WKHP≥35, WKW=1)
- Features: AGEP, SCHL, MARC (MAR==1), RELPC (RELP==0)
- Groups: sex (SEX==1 vs SEX==2), race (RAC1P==1 vs RAC1P≠1)
- Oracle gap $= R_g(f_0) - R_g(f_g^\ast)$, where $f_0$ is the constant predictor and $f_g^\ast$ is the group-specific OLS fit
- Gap ratio $=$ oracle gap of group 0 / oracle gap of group 1

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# ====== Comprehensive gap ratio distribution analysis ======
# Single-feature models across all 50 states × 2 partitions × 4 features

features_list = ['AGEP', 'SCHL', 'MARC', 'RELPC']

all_single_rows = []

for gb, g0n, g1n in [('sex', 'Male', 'Female'), ('race_binary', 'White', 'Non-White')]:
    df = pd.read_csv(f'../oracle_gap_scan_{gb}_employed.csv')
    for feat in features_list:
        feat_df = df[(df['n_features'] == 1) & (df['features'] == feat)]
        for _, r in feat_df.iterrows():
            all_single_rows.append({
                'state': r['state'],
                'group_by': gb,
                'feature': feat,
                'gap_g0': r['gap_g0'],
                'gap_g1': r['gap_g1'],
                'ratio': r['gap_ratio_g0_g1'],
                'baseline_g0': r['baseline_g0'],
                'baseline_g1': r['baseline_g1'],
                'oracle_g0': r['oracle_g0'],
                'oracle_g1': r['oracle_g1'],
            })

all_single = pd.DataFrame(all_single_rows)
print(f"Total single-feature configurations: {len(all_single)}")
print(f"  = 50 states × 2 partitions × 4 features = {50*2*4}")

Total single-feature configurations: 400
  = 50 states × 2 partitions × 4 features = 400


## Table 2: gap ratio distribution by partition and feature

In [3]:
# ====== Distribution summary by (partition × feature) ======

summary_rows = []

for gb in ['sex', 'race_binary']:
    for feat in features_list:
        subset = all_single[(all_single['group_by'] == gb) & (all_single['feature'] == feat)]
        ratios = subset['ratio']
        n = len(ratios)
        
        n_extreme = ((ratios < 0.5) | (ratios > 2.0)).sum()
        n_symmetric = ((ratios >= 0.8) & (ratios <= 1.25)).sum()
        
        summary_rows.append({
            'Partition': gb,
            'Feature': feat,
            'Min': ratios.min(),
            'Median': ratios.median(),
            'Max': ratios.max(),
            'Extreme (r<0.5 or r>2)': f'{n_extreme}/{n} ({100*n_extreme/n:.0f}%)',
            'Symmetric (0.8≤r≤1.25)': f'{n_symmetric}/{n} ({100*n_symmetric/n:.0f}%)',
        })

summary_df = pd.DataFrame(summary_rows)
print("Gap ratio distribution: single-feature models, 50 states")
print("=" * 90)
summary_df

Gap ratio distribution: single-feature models, 50 states


,Partition,Feature,Min,Median,Max,Extreme (r<0.5 or r>2),Symmetric (0.8≤r≤1.25)
0,sex,AGEP,0.5789,1.3169,2.0230,1/50 (2%),19/50 (38%)
1,sex,SCHL,0.3256,0.8756,1.3135,2/50 (4%),33/50 (66%)
2,sex,MARC,0.7089,1.6168,2.1780,7/50 (14%),4/50 (8%)
3,sex,RELPC,0.3651,1.0057,1.8071,3/50 (6%),26/50 (52%)
4,race_binary,AGEP,0.1010,0.6102,3.4810,17/50 (34%),10/50 (20%)
5,race_binary,SCHL,0.1453,0.6932,1.8387,6/50 (12%),12/50 (24%)
6,race_binary,MARC,0.1772,0.5279,2.9572,25/50 (50%),4/50 (8%)
7,race_binary,RELPC,0.0424,0.2964,2.8511,43/50 (86%),0/50 (0%)


## Section 6: all 400 configurations pooled

In [4]:
# ====== Overall distribution (all partitions × features pooled) ======
all_ratios = all_single['ratio']
n_total = len(all_ratios)

print("=" * 60)
print(f"OVERALL: {n_total} configurations (50 states × 2 partitions × 4 features)")
print("=" * 60)
print(f"Range:     [{all_ratios.min():.3f}, {all_ratios.max():.3f}]")
print(f"Median:    {all_ratios.median():.3f}")
print(f"Mean:      {all_ratios.mean():.3f}")
print(f"Std:       {all_ratios.std():.3f}")
print()

n_ext = ((all_ratios < 0.5) | (all_ratios > 2.0)).sum()
n_sym = ((all_ratios >= 0.8) & (all_ratios <= 1.25)).sum()
n_mod = n_total - n_ext - n_sym  # moderate asymmetry

print(f"Extreme asymmetry (r < 0.5 or r > 2.0):  {n_ext}/{n_total} ({100*n_ext/n_total:.1f}%)")
print(f"Moderate asymmetry:                       {n_mod}/{n_total} ({100*n_mod/n_total:.1f}%)")
print(f"Near-symmetric (0.8 ≤ r ≤ 1.25):          {n_sym}/{n_total} ({100*n_sym/n_total:.1f}%)")
print()
print("→ In the majority of configurations, groups differ substantially in predictability.")

OVERALL: 400 configurations (50 states × 2 partitions × 4 features)
Range:     [0.042, 3.481]
Median:    0.803
Mean:      0.889
Std:       0.511

Extreme asymmetry (r < 0.5 or r > 2.0):  104/400 (26.0%)
Moderate asymmetry:                       188/400 (47.0%)
Near-symmetric (0.8 ≤ r ≤ 1.25):          108/400 (27.0%)

→ In the majority of configurations, groups differ substantially in predictability.
